<a href="https://colab.research.google.com/github/Scangwski/-tennis-atp-data-warehouse/blob/main/Data_Management_Luca_Scanga.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Data Management

ATP Tennis Data Warehouse project.

Notebook structure:
1. setup and connection to the reconciled SQLite database;
2. data quality assessment;
3. cleaning rationale and analytical flags;
4. ETL from reconciled database to the star-schema data warehouse;
5. final validation;
6. export for Tableau Public;
7. conclusion.


## 1. Setup and reconciled database connection

The repository is cloned and the SQLite database connection is initialized. The function `q()` is used to run SQL queries and return pandas DataFrames.


In [1]:
!git clone https://github.com/Scangwski/-tennis-atp-data-warehouse.git

Cloning into '-tennis-atp-data-warehouse'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 15 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 3.27 MiB | 6.23 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [2]:
import sqlite3
import pandas as pd

db_path = "/content/-tennis-atp-data-warehouse/tennis_project.db"

conn = sqlite3.connect(db_path)

def q(sql):
    return pd.read_sql_query(sql, conn)

The reconciled database (`tennis_project.db`) was created and populated from the original `atp_tennis.csv` using DBeaver during the database construction step.  
In this notebook, the population of the reconciled database is verified before performing data quality assessment, cleaning decisions, and ETL loading into the data warehouse.

In [3]:
q("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
  AND name NOT LIKE 'sqlite_%'
ORDER BY name;
""")

,name
0,DIM_BEST_OF
1,DIM_COURT
2,DIM_DATE
3,DIM_PLAYER
4,DIM_ROUND
5,DIM_SURFACE
6,DIM_TOURNAMENT
7,FACT_MATCH
8,MATCHES
9,MATCH_DERIVED


## 2. Data quality assessment

The reconciled database is profiled through checks on population, uniqueness, completeness, consistency and numerical validity of the attributes used by the warehouse.


### 2.1 Reconciled database population check

The reconciled database has been populated from the original ATP tennis dataset.

The main transactional table `MATCHES` contains 67,512 records. The derived table `MATCH_DERIVED` also contains 67,512 records, meaning that each match has a corresponding row with derived analytical attributes.

At this stage, `FACT_MATCH` is empty because the data warehouse loading process has not been executed yet. It will be populated during the ETL phase.


In [4]:
q("""
SELECT 'PLAYER' AS table_name, COUNT(*) AS row_count FROM PLAYER
UNION ALL
SELECT 'TOURNAMENT', COUNT(*) FROM TOURNAMENT
UNION ALL
SELECT 'ROUND', COUNT(*) FROM ROUND
UNION ALL
SELECT 'MATCHES', COUNT(*) FROM MATCHES
UNION ALL
SELECT 'MATCH_DERIVED', COUNT(*) FROM MATCH_DERIVED
UNION ALL
SELECT 'FACT_MATCH', COUNT(*) FROM FACT_MATCH;
""")

,table_name,row_count
0,PLAYER,1798
1,TOURNAMENT,331
2,ROUND,8
3,MATCHES,67512
4,MATCH_DERIVED,67512
5,FACT_MATCH,0


In [5]:
q("""
SELECT COUNT(*) AS missing_derived_rows
FROM MATCHES m
LEFT JOIN MATCH_DERIVED md
    ON m.match_id = md.match_id
WHERE md.match_id IS NULL;
""")

,missing_derived_rows
0,0


In [6]:
q("""
SELECT COUNT(*) AS derived_rows_without_match
FROM MATCH_DERIVED md
LEFT JOIN MATCHES m
    ON md.match_id = m.match_id
WHERE m.match_id IS NULL;
""")

,derived_rows_without_match
0,0


The tables `MATCHES` and `MATCH_DERIVED` are correctly aligned: each match has one corresponding derived row and there are no orphan derived records.

### 2.2 Uniqueness of match identifiers

This check verifies that `match_id` uniquely identifies each match.

In [7]:
q("""
SELECT COUNT(*) AS duplicated_match_ids
FROM (
    SELECT match_id
    FROM MATCHES
    GROUP BY match_id
    HAVING COUNT(*) > 1
);
""")

,duplicated_match_ids
0,0


In [8]:
q("""
SELECT COUNT(*) AS duplicated_match_ids_in_derived
FROM (
    SELECT match_id
    FROM MATCH_DERIVED
    GROUP BY match_id
    HAVING COUNT(*) > 1
);
""")

,duplicated_match_ids_in_derived
0,0


No duplicated match identifiers were found. The `match_id` attribute can be used as a reliable identifier for the transactional fact.

### 2.3 Completeness of core match attributes

This check measures missing values in the main attributes required to identify and analyse a match.

In [9]:
q("""
SELECT
    SUM(CASE WHEN match_id IS NULL THEN 1 ELSE 0 END) AS null_match_id,
    SUM(CASE WHEN tournament_id IS NULL THEN 1 ELSE 0 END) AS null_tournament_id,
    SUM(CASE WHEN round_id IS NULL THEN 1 ELSE 0 END) AS null_round_id,
    SUM(CASE WHEN match_date IS NULL THEN 1 ELSE 0 END) AS null_match_date,
    SUM(CASE WHEN best_of IS NULL THEN 1 ELSE 0 END) AS null_best_of,
    SUM(CASE WHEN player_1_id IS NULL THEN 1 ELSE 0 END) AS null_player_1_id,
    SUM(CASE WHEN player_2_id IS NULL THEN 1 ELSE 0 END) AS null_player_2_id
FROM MATCHES;
""")

,null_match_id,null_tournament_id,null_round_id,null_match_date,null_best_of,null_player_1_id,null_player_2_id
0,0,0,0,0,0,0,0


In [10]:
q("""
SELECT
    SUM(CASE WHEN match_id IS NULL THEN 1 ELSE 0 END) AS null_match_id,
    SUM(CASE WHEN loser_id IS NULL THEN 1 ELSE 0 END) AS null_loser_id,
    SUM(CASE WHEN winner_rank IS NULL THEN 1 ELSE 0 END) AS null_winner_rank,
    SUM(CASE WHEN loser_rank IS NULL THEN 1 ELSE 0 END) AS null_loser_rank,
    SUM(CASE WHEN winner_points IS NULL THEN 1 ELSE 0 END) AS null_winner_points,
    SUM(CASE WHEN loser_points IS NULL THEN 1 ELSE 0 END) AS null_loser_points,
    SUM(CASE WHEN winner_odds IS NULL THEN 1 ELSE 0 END) AS null_winner_odds,
    SUM(CASE WHEN loser_odds IS NULL THEN 1 ELSE 0 END) AS null_loser_odds,
    SUM(CASE WHEN favorite_player_id IS NULL THEN 1 ELSE 0 END) AS null_favorite_player_id,
    SUM(CASE WHEN underdog_player_id IS NULL THEN 1 ELSE 0 END) AS null_underdog_player_id,
    SUM(CASE WHEN favorite_odds IS NULL THEN 1 ELSE 0 END) AS null_favorite_odds,
    SUM(CASE WHEN underdog_odds IS NULL THEN 1 ELSE 0 END) AS null_underdog_odds,
    SUM(CASE WHEN upset_flag IS NULL THEN 1 ELSE 0 END) AS null_upset_flag,
    SUM(CASE WHEN odd_gap IS NULL THEN 1 ELSE 0 END) AS null_odd_gap,
    SUM(CASE WHEN rank_gap IS NULL THEN 1 ELSE 0 END) AS null_rank_gap,
    SUM(CASE WHEN points_gap IS NULL THEN 1 ELSE 0 END) AS null_points_gap
FROM MATCH_DERIVED;
""")

,null_match_id,null_loser_id,null_winner_rank,null_loser_rank,null_winner_points,null_loser_points,null_winner_odds,null_loser_odds,null_favorite_player_id,null_underdog_player_id,null_favorite_odds,null_underdog_odds,null_upset_flag,null_odd_gap,null_rank_gap,null_points_gap
0,0,0,0,26,15652,15653,3783,3780,5211,5211,5211,5211,5211,5211,26,15653


In [11]:
q("""
SELECT
    COUNT(*) AS total_matches,
    SUM(CASE WHEN upset_flag IS NULL THEN 1 ELSE 0 END) AS missing_upset_flag,
    SUM(CASE WHEN winner_odds IS NULL OR loser_odds IS NULL THEN 1 ELSE 0 END) AS missing_odds_cases,
    SUM(CASE WHEN winner_odds = loser_odds THEN 1 ELSE 0 END) AS equal_odds_cases
FROM MATCH_DERIVED;
""")

,total_matches,missing_upset_flag,missing_odds_cases,equal_odds_cases
0,67512,5211,3783,1428


### 2.4 Missing ranking values analysis

This check investigates missing ranking-related attributes.  
Since `rank_gap` is derived from `winner_rank` and `loser_rank`, missing ranking values may propagate to the derived attribute.

In [12]:
q("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN winner_rank IS NULL THEN 1 ELSE 0 END) AS missing_winner_rank,
    SUM(CASE WHEN loser_rank IS NULL THEN 1 ELSE 0 END) AS missing_loser_rank,
    SUM(CASE WHEN rank_gap IS NULL THEN 1 ELSE 0 END) AS missing_rank_gap
FROM MATCH_DERIVED;
""")

,total_rows,missing_winner_rank,missing_loser_rank,missing_rank_gap
0,67512,0,26,26


In [13]:
q("""
SELECT
    SUM(CASE
        WHEN rank_gap IS NULL
         AND (winner_rank IS NULL OR loser_rank IS NULL)
        THEN 1 ELSE 0
    END) AS rank_gap_null_due_to_missing_rank,

    SUM(CASE
        WHEN rank_gap IS NULL
         AND winner_rank IS NOT NULL
         AND loser_rank IS NOT NULL
        THEN 1 ELSE 0
    END) AS unexplained_null_rank_gap
FROM MATCH_DERIVED;
""")

,rank_gap_null_due_to_missing_rank,unexplained_null_rank_gap
0,26,0


The missing values in `rank_gap` are explained by missing ranking information in the source attributes. Therefore, `rank_gap` does not present an independent data quality issue: the NULL value is propagated from the missing rank.

#### Matches with missing ranking values

The following query lists the matches where at least one ranking-related attribute is missing.

In [14]:
q("""
SELECT
    m.match_id,
    m.match_date,
    t.tournament_name,
    t.series,
    r.round_name,
    p1.player_name AS player_1,
    p2.player_name AS player_2,
    md.loser_id,
    md.winner_rank,
    md.loser_rank,
    md.rank_gap
FROM MATCHES m
JOIN MATCH_DERIVED md
    ON m.match_id = md.match_id
JOIN TOURNAMENT t
    ON m.tournament_id = t.tournament_id
JOIN ROUND r
    ON m.round_id = r.round_id
JOIN PLAYER p1
    ON m.player_1_id = p1.player_id
JOIN PLAYER p2
    ON m.player_2_id = p2.player_id
WHERE md.winner_rank IS NULL
   OR md.loser_rank IS NULL
   OR md.rank_gap IS NULL
ORDER BY m.match_date, m.match_id;
""")

,match_id,match_date,tournament_name,series,round_name,player_1,player_2,loser_id,winner_rank,loser_rank,rank_gap
0,64,2000-01-03,Qatar Open,International,1st Round,Al-Alawi S.K.,Berasategui A.,18,60,None,None
1,370,2000-02-14,Kroger St. Jude,International Gold,1st Round,Jensen L.,Mamiit C.,743,135,None,None
2,550,2000-02-28,Citrix Tennis Championships,International,1st Round,Roddick A.,Tieleman L.,1347,104,None,None
3,600,2000-03-06,Colombia Open,International,1st Round,Gonzalez P.,Etlis G.,600,137,None,None
4,602,2000-03-06,Colombia Open,International,1st Round,Hadad M.,Hantschk M.,640,105,None,None
5,607,2000-03-06,Colombia Open,International,1st Round,Puerta M.,Cortes J.,315,72,None,None
6,760,2000-03-23,Ericsson Open,Masters,2nd Round,Fish M.,Haas T.,498,21,None,None
7,1019,2000-05-01,Mallorca Open,International,1st Round,Bastl G.,Garcia M.,543,71,None,None
8,1805,2000-07-17,Croatia Open,International,1st Round,Damm M.,Peschek D.,1240,80,None,None
9,3440,2001-02-26,Dubai Championships,International Gold,1st Round,Eagle J.,Boutter J.,431,56,None,None


In [15]:
q("""
SELECT
    CAST(strftime('%Y', m.match_date) AS INTEGER) AS year,
    COUNT(*) AS missing_rank_matches
FROM MATCHES m
JOIN MATCH_DERIVED md
    ON m.match_id = md.match_id
WHERE md.winner_rank IS NULL
   OR md.loser_rank IS NULL
   OR md.rank_gap IS NULL
GROUP BY year
ORDER BY year;
""")

,year,missing_rank_matches
0,2000,9
1,2001,10
2,2002,6
3,2003,1


### 2.5 Missing ranking points analysis

This check investigates missing ranking points.  
Since `points_gap` is derived from `winner_points` and `loser_points`, missing source values may propagate to the derived attribute.

In [16]:
q("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN winner_points IS NULL THEN 1 ELSE 0 END) AS missing_winner_points,
    SUM(CASE WHEN loser_points IS NULL THEN 1 ELSE 0 END) AS missing_loser_points,
    SUM(CASE WHEN points_gap IS NULL THEN 1 ELSE 0 END) AS missing_points_gap
FROM MATCH_DERIVED;
""")

,total_rows,missing_winner_points,missing_loser_points,missing_points_gap
0,67512,15652,15653,15653


In [17]:
q("""
SELECT
    SUM(CASE
        WHEN points_gap IS NULL
         AND (winner_points IS NULL OR loser_points IS NULL)
        THEN 1 ELSE 0
    END) AS points_gap_null_due_to_missing_points,

    SUM(CASE
        WHEN points_gap IS NULL
         AND winner_points IS NOT NULL
         AND loser_points IS NOT NULL
        THEN 1 ELSE 0
    END) AS unexplained_null_points_gap
FROM MATCH_DERIVED;
""")

,points_gap_null_due_to_missing_points,unexplained_null_points_gap
0,15653,0


The missing values in `points_gap` are caused by missing ranking points in the source attributes. The NULL values are therefore preserved instead of being automatically imputed, because artificial ranking points could distort the analysis.

#### Matches with missing ranking points

The following query extracts the matches where at least one ranking-points-related attribute is missing.  
Since the number of affected records is high, only a sample is displayed in the notebook, while the full result is saved as a CSV file.

In [18]:
missing_points_matches = q("""
SELECT
    m.match_id,
    m.match_date,
    t.tournament_name,
    t.series,
    r.round_name,
    p1.player_name AS player_1,
    p2.player_name AS player_2,
    md.loser_id,
    md.winner_points,
    md.loser_points,
    md.points_gap
FROM MATCHES m
JOIN MATCH_DERIVED md
    ON m.match_id = md.match_id
JOIN TOURNAMENT t
    ON m.tournament_id = t.tournament_id
JOIN ROUND r
    ON m.round_id = r.round_id
JOIN PLAYER p1
    ON m.player_1_id = p1.player_id
JOIN PLAYER p2
    ON m.player_2_id = p2.player_id
WHERE md.winner_points IS NULL
   OR md.loser_points IS NULL
   OR md.points_gap IS NULL
ORDER BY m.match_date, m.match_id;
""")

missing_points_matches.to_csv("missing_points_matches.csv", index=False)

missing_points_matches.head(20)

,match_id,match_date,tournament_name,series,round_name,player_1,player_2,loser_id,winner_points,loser_points,points_gap
0,1,2000-01-03,Australian Hardcourt Championships,International,1st Round,Dosedel S.,Ljubicic I.,927,NaN,None,None
1,2,2000-01-03,Australian Hardcourt Championships,International,1st Round,Clement A.,Enqvist T.,298,NaN,None,None
2,3,2000-01-03,Australian Hardcourt Championships,International,1st Round,Escude N.,Baccanello P.,80,NaN,None,None
3,4,2000-01-03,Australian Hardcourt Championships,International,1st Round,Knippschild J.,Federer R.,810,NaN,None,None
4,5,2000-01-03,Australian Hardcourt Championships,International,1st Round,Fromberg R.,Woodbridge T.,1738,NaN,None,None
5,6,2000-01-03,Australian Hardcourt Championships,International,1st Round,Arthurs W.,Gambill J.M.,66,NaN,None,None
6,7,2000-01-03,Australian Hardcourt Championships,International,1st Round,Grosjean S.,Ilie A.,716,NaN,None,None
7,8,2000-01-03,Australian Hardcourt Championships,International,1st Round,Balcells J.,Henman T.,93,NaN,None,None
8,9,2000-01-03,Australian Hardcourt Championships,International,1st Round,Hewitt L.,Woodforde M.,1739,NaN,None,None
9,10,2000-01-03,Australian Hardcourt Championships,International,1st Round,Tebbutt M.,Lisnard J.,1566,NaN,None,None


In [19]:
q("""
SELECT
    CAST(strftime('%Y', m.match_date) AS INTEGER) AS year,
    COUNT(*) AS missing_points_matches
FROM MATCHES m
JOIN MATCH_DERIVED md
    ON m.match_id = md.match_id
WHERE md.winner_points IS NULL
   OR md.loser_points IS NULL
   OR md.points_gap IS NULL
GROUP BY year
ORDER BY year;
""")

,year,missing_points_matches
0,2000,2874
1,2001,2979
2,2002,2700
3,2003,2698
4,2004,2782
5,2005,1619
6,2006,1


In [20]:
q("""
SELECT
    CAST(strftime('%Y', m.match_date) AS INTEGER) AS year,
    COUNT(*) AS missing_rank_matches
FROM MATCHES m
JOIN MATCH_DERIVED md
    ON m.match_id = md.match_id
WHERE md.winner_rank IS NULL
   OR md.loser_rank IS NULL
   OR md.rank_gap IS NULL
GROUP BY year
ORDER BY year;
""")

,year,missing_rank_matches
0,2000,9
1,2001,10
2,2002,6
3,2003,1


In [21]:
q("""
SELECT
    CAST(strftime('%Y', m.match_date) AS INTEGER) AS year,
    COUNT(*) AS missing_points_matches
FROM MATCHES m
JOIN MATCH_DERIVED md
    ON m.match_id = md.match_id
WHERE md.winner_points IS NULL
   OR md.loser_points IS NULL
   OR md.points_gap IS NULL
GROUP BY year
ORDER BY year;
""")

,year,missing_points_matches
0,2000,2874
1,2001,2979
2,2002,2700
3,2003,2698
4,2004,2782
5,2005,1619
6,2006,1


In [22]:
q("""
SELECT
    t.series,
    COUNT(*) AS missing_points_matches
FROM MATCHES m
JOIN MATCH_DERIVED md
    ON m.match_id = md.match_id
JOIN TOURNAMENT t
    ON m.tournament_id = t.tournament_id
WHERE md.winner_points IS NULL
   OR md.loser_points IS NULL
   OR md.points_gap IS NULL
GROUP BY t.series
ORDER BY missing_points_matches DESC;
""")

,series,missing_points_matches
0,International,7417
1,Masters,3158
2,Grand Slam,2801
3,International Gold,2202
4,Masters Cup,75


In [23]:
q("""
SELECT
    m.surface,
    COUNT(*) AS missing_points_matches
FROM MATCHES m
JOIN MATCH_DERIVED md
    ON m.match_id = md.match_id
WHERE md.winner_points IS NULL
   OR md.loser_points IS NULL
   OR md.points_gap IS NULL
GROUP BY m.surface
ORDER BY missing_points_matches DESC;
""")

,surface,missing_points_matches
0,Hard,7597
1,Clay,5449
2,Grass,1752
3,Carpet,855


In [24]:
q("""
SELECT
    m.court,
    COUNT(*) AS missing_points_matches
FROM MATCHES m
JOIN MATCH_DERIVED md
    ON m.match_id = md.match_id
WHERE md.winner_points IS NULL
   OR md.loser_points IS NULL
   OR md.points_gap IS NULL
GROUP BY m.court
ORDER BY missing_points_matches DESC;
""")

,court,missing_points_matches
0,Outdoor,13021
1,Indoor,2632


### 2.6 Missing betting odds analysis

This check investigates the availability of betting odds in `MATCH_DERIVED`.  
Odds-related derived attributes can only be computed when both winner and loser odds are available.

In [25]:
q("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN winner_odds IS NULL THEN 1 ELSE 0 END) AS missing_winner_odds,
    SUM(CASE WHEN loser_odds IS NULL THEN 1 ELSE 0 END) AS missing_loser_odds,
    SUM(CASE WHEN winner_odds IS NULL OR loser_odds IS NULL THEN 1 ELSE 0 END) AS at_least_one_missing_odd,
    SUM(CASE WHEN winner_odds IS NOT NULL AND loser_odds IS NOT NULL THEN 1 ELSE 0 END) AS both_odds_available
FROM MATCH_DERIVED;
""")

,total_rows,missing_winner_odds,missing_loser_odds,at_least_one_missing_odd,both_odds_available
0,67512,3783,3780,3783,63729


In [26]:
q("""
SELECT
    SUM(CASE
        WHEN winner_odds IS NOT NULL
         AND loser_odds IS NOT NULL
         AND winner_odds = loser_odds
        THEN 1 ELSE 0
    END) AS equal_odds_cases
FROM MATCH_DERIVED;
""")

,equal_odds_cases
0,1428


The betting odds completeness check shows that the dataset contains 67,512 matches.  
`winner_odds` is missing in 3,783 records, while `loser_odds` is missing in 3,780 records.

The number of matches with at least one missing odd is 3,783. Therefore, in these records the betting information is not fully available.

The check also identifies 1,428 matches where both odds are available but `winner_odds = loser_odds`. In these cases, the odds are present but they do not allow the identification of a clear favorite and a clear underdog.

As a result, odds-related derived attributes must be treated carefully: they can only be considered reliable when both odds are available and different.

### 2.7 Missing derived odds attributes

This check verifies whether missing values in favorite/underdog attributes are explained by missing odds or equal odds.

In [27]:
q("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN favorite_player_id IS NULL THEN 1 ELSE 0 END) AS missing_favorite_player_id,
    SUM(CASE WHEN underdog_player_id IS NULL THEN 1 ELSE 0 END) AS missing_underdog_player_id,
    SUM(CASE WHEN favorite_odds IS NULL THEN 1 ELSE 0 END) AS missing_favorite_odds,
    SUM(CASE WHEN underdog_odds IS NULL THEN 1 ELSE 0 END) AS missing_underdog_odds,
    SUM(CASE WHEN odd_gap IS NULL THEN 1 ELSE 0 END) AS missing_odd_gap,

    SUM(CASE
        WHEN winner_odds IS NULL OR loser_odds IS NULL
        THEN 1 ELSE 0
    END) AS due_to_missing_odds,

    SUM(CASE
        WHEN winner_odds IS NOT NULL
         AND loser_odds IS NOT NULL
         AND winner_odds = loser_odds
        THEN 1 ELSE 0
    END) AS due_to_equal_odds
FROM MATCH_DERIVED;
""")

,total_rows,missing_favorite_player_id,missing_underdog_player_id,missing_favorite_odds,missing_underdog_odds,missing_odd_gap,due_to_missing_odds,due_to_equal_odds
0,67512,5211,5211,5211,5211,5211,3783,1428


The check shows that the odds-derived attributes have 5,211 missing values:

- `favorite_player_id`
- `underdog_player_id`
- `favorite_odds`
- `underdog_odds`
- `odd_gap`

These missing values are fully explained by two conditions:

- 3,783 records have at least one missing odd;
- 1,428 records have equal odds for winner and loser.

Therefore:

3,783 + 1,428 = 5,211

This means that the missing values in favorite/underdog attributes are not independent data quality errors. They are expected consequences of cases where the betting odds do not allow a reliable derivation of favorite and underdog information.

In [28]:
q("""
SELECT
    SUM(CASE
        WHEN favorite_player_id IS NULL
         AND winner_odds IS NOT NULL
         AND loser_odds IS NOT NULL
         AND winner_odds <> loser_odds
        THEN 1 ELSE 0
    END) AS unexplained_missing_favorite,

    SUM(CASE
        WHEN underdog_player_id IS NULL
         AND winner_odds IS NOT NULL
         AND loser_odds IS NOT NULL
         AND winner_odds <> loser_odds
        THEN 1 ELSE 0
    END) AS unexplained_missing_underdog,

    SUM(CASE
        WHEN odd_gap IS NULL
         AND winner_odds IS NOT NULL
         AND loser_odds IS NOT NULL
         AND winner_odds <> loser_odds
        THEN 1 ELSE 0
    END) AS unexplained_missing_odd_gap
FROM MATCH_DERIVED;
""")

,unexplained_missing_favorite,unexplained_missing_underdog,unexplained_missing_odd_gap
0,0,0,0


The check returns 0 unexplained missing values.  
This confirms that favorite, underdog and odd gap are missing only when the required betting information is not sufficient to derive them reliably.

### 2.8 Consistency of player identifiers in derived attributes

This check verifies that the player identifiers stored in `MATCH_DERIVED` are consistent with the players involved in each match.

In particular, the check verifies that:

- `loser_id` is one of the two players of the match;
- `favorite_player_id`, when available, is one of the two players of the match;
- `underdog_player_id`, when available, is one of the two players of the match;
- favorite and underdog are not the same player.

In [29]:
q("""
SELECT COUNT(*) AS invalid_loser_id
FROM MATCHES m
JOIN MATCH_DERIVED md
    ON m.match_id = md.match_id
WHERE md.loser_id NOT IN (m.player_1_id, m.player_2_id);
""")

,invalid_loser_id
0,0


In [30]:
q("""
SELECT COUNT(*) AS invalid_favorite_player_id
FROM MATCHES m
JOIN MATCH_DERIVED md
    ON m.match_id = md.match_id
WHERE md.favorite_player_id IS NOT NULL
  AND md.favorite_player_id NOT IN (m.player_1_id, m.player_2_id);
""")

,invalid_favorite_player_id
0,0


In [31]:
q("""
SELECT COUNT(*) AS invalid_underdog_player_id
FROM MATCHES m
JOIN MATCH_DERIVED md
    ON m.match_id = md.match_id
WHERE md.underdog_player_id IS NOT NULL
  AND md.underdog_player_id NOT IN (m.player_1_id, m.player_2_id);
""")

,invalid_underdog_player_id
0,0


In [32]:
q("""
SELECT COUNT(*) AS same_favorite_and_underdog
FROM MATCH_DERIVED
WHERE favorite_player_id IS NOT NULL
  AND underdog_player_id IS NOT NULL
  AND favorite_player_id = underdog_player_id;
""")

,same_favorite_and_underdog
0,0


All consistency checks return 0 invalid records.

This means that:

- `loser_id` always refers to one of the two players involved in the match;
- `favorite_player_id`, when available, always refers to one of the two players involved in the match;
- `underdog_player_id`, when available, always refers to one of the two players involved in the match;
- favorite and underdog are never assigned to the same player.

Therefore, the player identifiers in `MATCH_DERIVED` are consistent with the original match information stored in `MATCHES`.

### 2.9 Validity of numerical attributes

This check detects impossible numerical values, such as negative rankings, negative ranking points or non-positive betting odds.

In [33]:
q("""
SELECT
    SUM(CASE WHEN winner_rank < 0 THEN 1 ELSE 0 END) AS negative_winner_rank,
    SUM(CASE WHEN loser_rank < 0 THEN 1 ELSE 0 END) AS negative_loser_rank,
    SUM(CASE WHEN winner_points < 0 THEN 1 ELSE 0 END) AS negative_winner_points,
    SUM(CASE WHEN loser_points < 0 THEN 1 ELSE 0 END) AS negative_loser_points,
    SUM(CASE WHEN winner_odds <= 0 THEN 1 ELSE 0 END) AS non_positive_winner_odds,
    SUM(CASE WHEN loser_odds <= 0 THEN 1 ELSE 0 END) AS non_positive_loser_odds,
    SUM(CASE WHEN favorite_odds <= 0 THEN 1 ELSE 0 END) AS non_positive_favorite_odds,
    SUM(CASE WHEN underdog_odds <= 0 THEN 1 ELSE 0 END) AS non_positive_underdog_odds
FROM MATCH_DERIVED;
""")

,negative_winner_rank,negative_loser_rank,negative_winner_points,negative_loser_points,non_positive_winner_odds,non_positive_loser_odds,non_positive_favorite_odds,non_positive_underdog_odds
0,0,0,0,0,1,1,0,0


In [34]:
q("""
SELECT
    m.match_id,
    m.match_date,
    t.tournament_name,
    t.series,
    m.surface,
    m.court,
    r.round_name,
    p1.player_name AS player_1,
    p2.player_name AS player_2,
    md.winner_odds,
    md.loser_odds,
    md.favorite_odds,
    md.underdog_odds,
    md.odd_gap
FROM MATCHES m
JOIN MATCH_DERIVED md
    ON m.match_id = md.match_id
JOIN TOURNAMENT t
    ON m.tournament_id = t.tournament_id
JOIN ROUND r
    ON m.round_id = r.round_id
JOIN PLAYER p1
    ON m.player_1_id = p1.player_id
JOIN PLAYER p2
    ON m.player_2_id = p2.player_id
WHERE md.loser_odds <= 0
   OR md.winner_odds <= 0
   OR md.favorite_odds <= 0
   OR md.underdog_odds <= 0;
""")

,match_id,match_date,tournament_name,series,surface,court,round_name,player_1,player_2,winner_odds,loser_odds,favorite_odds,underdog_odds,odd_gap
0,65824,2025-07-22,Croatia Open,ATP250,Clay,Outdoor,2nd Round,Atmane T.,Llamas Ruiz P.,0.0,0.0,None,None,None


The numerical validity check shows that rankings and ranking points do not contain negative values. Most betting odds are also valid.

However, one record contains non-positive betting odds. The affected match is `match_id = 65824`, played on 2025-07-22 at the Croatia Open between Atmane T. and Llamas Ruiz P.

For this match, both `winner_odds` and `loser_odds` are equal to 0.0. Since betting odds must be positive values, these values are not considered valid odds. They are interpreted as invalid or placeholder values rather than meaningful betting information.

The match itself is not removed from the dataset, because it is a valid tennis match. However, its odds are treated as not reliable and should be excluded from analyses based on betting odds.

In [35]:
q("""
SELECT best_of, COUNT(*) AS occurrences
FROM MATCHES
GROUP BY best_of
ORDER BY best_of;
""")

,best_of,occurrences
0,3,54806
1,5,12706


The domain check on `best_of` shows only two values: 3 and 5.  
This is consistent with ATP tennis matches, where matches are generally played as best-of-three or best-of-five sets.

Therefore, no invalid values were detected for the `best_of` attribute.

### 2.10 Summary of data quality findings

The assessment shows that the reconciled database is structurally suitable for the data warehouse: `MATCHES` and `MATCH_DERIVED` are aligned, `match_id` is unique, core attributes are complete, and player identifiers in the derived table are consistent with the players involved in each match.

The relevant issues concern analytical attributes rather than the validity of the match records. Ranking and points values may be missing, and 5,211 matches are not reliable for odds-based upset analysis: 3,783 have at least one missing odd, 1,427 have valid equal odds, and 1 match has invalid odds equal to 0.0.

No valid match record is removed as a result of these findings.


## 3. Cleaning rationale and analytical flags

The cleaning strategy preserves NULL values and avoids imputing rankings, ranking points or betting odds. Imputation would introduce artificial values and could distort rank-gap, points-gap, odds-gap and upset-rate analyses.

Odds-based upset analysis is limited to matches where a reliable favorite/underdog distinction exists. The main KPI is calculated as:

`upset_rate = SUM(upset_count) / SUM(observable_upset_count)`

rather than over the total number of matches. Non-observable cases are preserved in the warehouse and identified through analytical flags and measures.


In [36]:
def exec_sql(sql):
    conn.executescript(sql)
    conn.commit()

In [37]:
exec_sql("""
DROP VIEW IF EXISTS V_MATCH_ANALYTICAL_FLAGS;

CREATE VIEW V_MATCH_ANALYTICAL_FLAGS AS
SELECT
    m.match_id,
    m.match_date,
    CAST(strftime('%Y%m%d', m.match_date) AS INTEGER) AS date_key,

    m.tournament_id,
    m.round_id,
    m.player_1_id,
    m.player_2_id,

    CASE
        WHEN md.loser_id = m.player_1_id THEN m.player_2_id
        ELSE m.player_1_id
    END AS winner_player_id,

    md.loser_id AS loser_player_id,

    md.favorite_player_id,
    md.underdog_player_id,

    m.best_of,
    m.court,
    m.surface,

    md.winner_rank,
    md.loser_rank,
    md.rank_gap,

    md.winner_points,
    md.loser_points,
    md.points_gap,

    md.winner_odds,
    md.loser_odds,
    md.favorite_odds,
    md.underdog_odds,
    md.odd_gap,

    1 AS match_count,

    CASE
        WHEN md.winner_rank IS NOT NULL
         AND md.loser_rank IS NOT NULL
        THEN 1 ELSE 0
    END AS rank_observable_flag,

    CASE
        WHEN md.winner_points IS NOT NULL
         AND md.loser_points IS NOT NULL
        THEN 1 ELSE 0
    END AS points_observable_flag,

    CASE
        WHEN md.winner_odds IS NULL
          OR md.loser_odds IS NULL
        THEN 1 ELSE 0
    END AS missing_odds_flag,

    CASE
        WHEN md.winner_odds <= 0
          OR md.loser_odds <= 0
        THEN 1 ELSE 0
    END AS invalid_odds_flag,

    CASE
        WHEN md.winner_odds > 0
         AND md.loser_odds > 0
         AND md.winner_odds = md.loser_odds
        THEN 1 ELSE 0
    END AS equal_odds_flag,

    CASE
        WHEN md.winner_odds > 0
         AND md.loser_odds > 0
         AND md.winner_odds <> md.loser_odds
        THEN 1 ELSE 0
    END AS observable_upset_count,

    CASE
        WHEN md.winner_odds > 0
         AND md.loser_odds > 0
         AND md.winner_odds <> md.loser_odds
         AND md.underdog_player_id =
            CASE
                WHEN md.loser_id = m.player_1_id THEN m.player_2_id
                ELSE m.player_1_id
            END
        THEN 1 ELSE 0
    END AS upset_count,

    CASE
        WHEN md.winner_odds > 0
         AND md.loser_odds > 0
         AND md.winner_odds <> md.loser_odds
         AND md.favorite_player_id =
            CASE
                WHEN md.loser_id = m.player_1_id THEN m.player_2_id
                ELSE m.player_1_id
            END
        THEN 1 ELSE 0
    END AS non_upset_count,

    CASE
        WHEN NOT (
            md.winner_odds > 0
            AND md.loser_odds > 0
            AND md.winner_odds <> md.loser_odds
        )
        THEN 1 ELSE 0
    END AS missing_upset_flag_count

FROM MATCHES m
JOIN MATCH_DERIVED md
    ON m.match_id = md.match_id;
""")

In [38]:
q("""
SELECT
    COUNT(*) AS total_matches,
    SUM(match_count) AS match_count,
    SUM(missing_odds_flag) AS missing_odds_matches,
    SUM(equal_odds_flag) AS equal_odds_matches,
    SUM(invalid_odds_flag) AS invalid_odds_matches,
    SUM(observable_upset_count) AS observable_upset_matches,
    SUM(upset_count) AS upset_matches,
    SUM(non_upset_count) AS non_upset_matches,
    SUM(missing_upset_flag_count) AS non_observable_upset_matches
FROM V_MATCH_ANALYTICAL_FLAGS;
""")

,total_matches,match_count,missing_odds_matches,equal_odds_matches,invalid_odds_matches,observable_upset_matches,upset_matches,non_upset_matches,non_observable_upset_matches
0,67512,67512,3783,1427,1,62301,18607,43694,1428


## 4. ETL from reconciled database to data warehouse

The ETL process loads the dimensional data warehouse from the reconciled database. The fact table keeps the transactional granularity: one row per tennis match.

### 4.1 Create/verify star schema


In [39]:
def show_schema(table_name):
    return q(f"PRAGMA table_info({table_name});")

for table in [
    "DIM_DATE", "DIM_TOURNAMENT", "DIM_ROUND", "DIM_SURFACE",
    "DIM_COURT", "DIM_BEST_OF", "DIM_PLAYER", "FACT_MATCH"
]:
    print(f"\n--- {table} ---")
    display(show_schema(table))



--- DIM_DATE ---


,cid,name,type,notnull,dflt_value,pk
0,0,date_key,INTEGER,0,None,1
1,1,match_date,TEXT,1,None,0
2,2,month,INTEGER,0,None,0
3,3,year,INTEGER,0,None,0



--- DIM_TOURNAMENT ---


,cid,name,type,notnull,dflt_value,pk
0,0,tournament_key,INTEGER,0,None,1
1,1,tournament_id,INTEGER,0,None,0
2,2,tournament_name,TEXT,0,None,0
3,3,series,TEXT,0,None,0



--- DIM_ROUND ---


...

,cid,name,type,notnull,dflt_value,pk
0,0,best_of_key,INTEGER,0,None,1
1,1,best_of,INTEGER,0,None,0



--- DIM_PLAYER ---


,cid,name,type,notnull,dflt_value,pk
0,0,player_key,INTEGER,0,None,1
1,1,player_id,INTEGER,0,None,0
2,2,player_name,TEXT,0,None,0



--- FACT_MATCH ---


,cid,name,type,notnull,dflt_value,pk
0,0,match_key,INTEGER,0,None,1
1,1,date_key,INTEGER,0,None,0
2,2,tournament_key,INTEGER,0,None,0
3,3,round_key,INTEGER,0,None,0
4,4,surface_key,INTEGER,0,None,0
5,5,court_key,INTEGER,0,None,0
6,6,best_of_key,INTEGER,0,None,0
7,7,winner_player_key,INTEGER,0,None,0
8,8,loser_player_key,INTEGER,0,None,0
9,9,favorite_player_key,INTEGER,0,None,0


### 4.2 Clear previous DW data

The warehouse tables are cleared before loading, so the ETL process can be re-executed without duplicating rows.


In [40]:
exec_sql("""
DELETE FROM FACT_MATCH;

DELETE FROM DIM_DATE;
DELETE FROM DIM_TOURNAMENT;
DELETE FROM DIM_ROUND;
DELETE FROM DIM_SURFACE;
DELETE FROM DIM_COURT;
DELETE FROM DIM_BEST_OF;
DELETE FROM DIM_PLAYER;
""")


In [41]:
q("""
SELECT 'DIM_DATE' AS table_name, COUNT(*) AS rows FROM DIM_DATE
UNION ALL
SELECT 'DIM_TOURNAMENT', COUNT(*) FROM DIM_TOURNAMENT
UNION ALL
SELECT 'DIM_ROUND', COUNT(*) FROM DIM_ROUND
UNION ALL
SELECT 'DIM_PLAYER', COUNT(*) FROM DIM_PLAYER
UNION ALL
SELECT 'DIM_SURFACE', COUNT(*) FROM DIM_SURFACE
UNION ALL
SELECT 'DIM_COURT', COUNT(*) FROM DIM_COURT
UNION ALL
SELECT 'DIM_BEST_OF', COUNT(*) FROM DIM_BEST_OF;
""")

,table_name,rows
0,DIM_DATE,0
1,DIM_TOURNAMENT,0
2,DIM_ROUND,0
3,DIM_PLAYER,0
4,DIM_SURFACE,0
5,DIM_COURT,0
6,DIM_BEST_OF,0


### 4.3 Load dimensions

Dimension tables are populated from the reconciled database and from the analytical view.


In [42]:
def table_columns(table_name):
    return list(pd.read_sql_query(f"PRAGMA table_info({table_name});", conn)["name"])

def append_matching_columns(df, table_name):
    cols = table_columns(table_name)
    common_cols = [c for c in df.columns if c in cols]

    if not common_cols:
        raise ValueError(f"No matching columns found for {table_name}")

    df[common_cols].drop_duplicates().to_sql(
        table_name,
        conn,
        if_exists="append",
        index=False
    )

    print(f"Loaded {len(df[common_cols].drop_duplicates())} rows into {table_name}")
    print("Columns loaded:", common_cols)

In [43]:
dim_date = q("""
SELECT DISTINCT
    CAST(strftime('%Y%m%d', match_date) AS INTEGER) AS date_key,
    CAST(strftime('%Y%m%d', match_date) AS INTEGER) AS date_id,
    match_date,
    CAST(strftime('%d', match_date) AS INTEGER) AS day,
    CAST(strftime('%m', match_date) AS INTEGER) AS month,
    CAST(strftime('%Y', match_date) AS INTEGER) AS year,
    CASE
        WHEN CAST(strftime('%m', match_date) AS INTEGER) BETWEEN 1 AND 3 THEN 1
        WHEN CAST(strftime('%m', match_date) AS INTEGER) BETWEEN 4 AND 6 THEN 2
        WHEN CAST(strftime('%m', match_date) AS INTEGER) BETWEEN 7 AND 9 THEN 3
        ELSE 4
    END AS quarter,
    strftime('%w', match_date) AS day_of_week
FROM MATCHES
WHERE match_date IS NOT NULL;
""")

append_matching_columns(dim_date, "DIM_DATE")

Loaded 6609 rows into DIM_DATE
Columns loaded: ['date_key', 'match_date', 'month', 'year']


In [44]:
dim_player = q("""
SELECT DISTINCT
    player_id,
    player_name
FROM PLAYER;
""")

append_matching_columns(dim_player, "DIM_PLAYER")

Loaded 1798 rows into DIM_PLAYER
Columns loaded: ['player_id', 'player_name']


In [45]:
dim_round = q("""
SELECT DISTINCT
    round_id,
    round_name,
    round_order
FROM ROUND;
""")

append_matching_columns(dim_round, "DIM_ROUND")

Loaded 8 rows into DIM_ROUND
Columns loaded: ['round_id', 'round_name', 'round_order']


In [46]:
dim_tournament = q("""
SELECT DISTINCT
    t.tournament_id,
    t.tournament_name,
    t.series,
    m.court,
    m.surface
FROM TOURNAMENT t
JOIN MATCHES m
    ON t.tournament_id = m.tournament_id;
""")

append_matching_columns(dim_tournament, "DIM_TOURNAMENT")

Loaded 331 rows into DIM_TOURNAMENT
Columns loaded: ['tournament_id', 'tournament_name', 'series']


In [47]:
# Load small categorical dimensions that are modeled separately in the star schema.
exec_sql("""
INSERT INTO DIM_SURFACE (surface_key, surface)
SELECT
    ROW_NUMBER() OVER (ORDER BY surface) AS surface_key,
    surface
FROM (
    SELECT DISTINCT surface
    FROM V_MATCH_ANALYTICAL_FLAGS
    WHERE surface IS NOT NULL
);

INSERT INTO DIM_COURT (court_key, court)
SELECT
    ROW_NUMBER() OVER (ORDER BY court) AS court_key,
    court
FROM (
    SELECT DISTINCT court
    FROM V_MATCH_ANALYTICAL_FLAGS
    WHERE court IS NOT NULL
);

INSERT INTO DIM_BEST_OF (best_of_key, best_of)
SELECT
    ROW_NUMBER() OVER (ORDER BY best_of) AS best_of_key,
    best_of
FROM (
    SELECT DISTINCT best_of
    FROM V_MATCH_ANALYTICAL_FLAGS
    WHERE best_of IS NOT NULL
);
""")

In [48]:
q("""
SELECT 'DIM_DATE' AS table_name, COUNT(*) AS row_count FROM DIM_DATE
UNION ALL
SELECT 'DIM_TOURNAMENT', COUNT(*) FROM DIM_TOURNAMENT
UNION ALL
SELECT 'DIM_ROUND', COUNT(*) FROM DIM_ROUND
UNION ALL
SELECT 'DIM_SURFACE', COUNT(*) FROM DIM_SURFACE
UNION ALL
SELECT 'DIM_COURT', COUNT(*) FROM DIM_COURT
UNION ALL
SELECT 'DIM_BEST_OF', COUNT(*) FROM DIM_BEST_OF
UNION ALL
SELECT 'DIM_PLAYER', COUNT(*) FROM DIM_PLAYER;
""")


,table_name,row_count
0,DIM_DATE,6609
1,DIM_TOURNAMENT,331
2,DIM_ROUND,8
3,DIM_SURFACE,4
4,DIM_COURT,2
5,DIM_BEST_OF,2
6,DIM_PLAYER,1798


### 4.4 Load FACT_MATCH

The fact table is loaded by resolving source identifiers and categorical values into the surrogate keys of the dimensional tables.


In [49]:
# Load FACT_MATCH by resolving operational IDs/values into dimensional surrogate keys.
exec_sql("""
INSERT INTO FACT_MATCH (
    match_key,
    date_key,
    tournament_key,
    round_key,
    surface_key,
    court_key,
    best_of_key,
    winner_player_key,
    loser_player_key,
    favorite_player_key,
    underdog_player_key,

    match_count,
    upset_count,
    non_upset_count,
    observable_upset_count,
    missing_upset_flag_count,
    equal_odds_count,

    winner_rank,
    loser_rank,
    rank_gap,
    winner_points,
    loser_points,
    points_gap,
    winner_odds,
    loser_odds,
    favorite_odds,
    underdog_odds,
    odd_gap
)
SELECT
    v.match_id AS match_key,
    v.date_key,

    dt.tournament_key,
    dr.round_key,
    ds.surface_key,
    dc.court_key,
    db.best_of_key,

    wp.player_key AS winner_player_key,
    lp.player_key AS loser_player_key,
    fp.player_key AS favorite_player_key,
    up.player_key AS underdog_player_key,

    v.match_count,
    v.upset_count,
    v.non_upset_count,
    v.observable_upset_count,

    CASE
        WHEN fp.player_key IS NULL OR up.player_key IS NULL
        THEN 1 ELSE 0
    END AS missing_upset_flag_count,

    v.equal_odds_flag AS equal_odds_count,

    v.winner_rank,
    v.loser_rank,
    v.rank_gap,
    v.winner_points,
    v.loser_points,
    v.points_gap,
    v.winner_odds,
    v.loser_odds,
    v.favorite_odds,
    v.underdog_odds,
    v.odd_gap

FROM V_MATCH_ANALYTICAL_FLAGS v

LEFT JOIN DIM_TOURNAMENT dt
    ON v.tournament_id = dt.tournament_id

LEFT JOIN DIM_ROUND dr
    ON v.round_id = dr.round_id

LEFT JOIN DIM_SURFACE ds
    ON v.surface = ds.surface

LEFT JOIN DIM_COURT dc
    ON v.court = dc.court

LEFT JOIN DIM_BEST_OF db
    ON v.best_of = db.best_of

LEFT JOIN DIM_PLAYER wp
    ON v.winner_player_id = wp.player_id

LEFT JOIN DIM_PLAYER lp
    ON v.loser_player_id = lp.player_id

LEFT JOIN DIM_PLAYER fp
    ON v.favorite_player_id = fp.player_id

LEFT JOIN DIM_PLAYER up
    ON v.underdog_player_id = up.player_id;
""")

## 5. Final validation

### 5.1 Fact row count

The number of fact rows must match the number of matches in the reconciled database.


In [50]:
q("""
SELECT
    (SELECT COUNT(*) FROM MATCHES) AS reconciled_matches,
    (SELECT COUNT(*) FROM FACT_MATCH) AS fact_matches,
    (SELECT COUNT(*) FROM MATCHES) - (SELECT COUNT(*) FROM FACT_MATCH) AS difference;
""")

,reconciled_matches,fact_matches,difference
0,67512,67512,0


### 5.2 Dimension row counts

This check reports the number of rows loaded in each dimension and in the fact table.


In [51]:
q("""
SELECT 'DIM_DATE' AS table_name, COUNT(*) AS row_count FROM DIM_DATE
UNION ALL
SELECT 'DIM_TOURNAMENT', COUNT(*) FROM DIM_TOURNAMENT
UNION ALL
SELECT 'DIM_ROUND', COUNT(*) FROM DIM_ROUND
UNION ALL
SELECT 'DIM_SURFACE', COUNT(*) FROM DIM_SURFACE
UNION ALL
SELECT 'DIM_COURT', COUNT(*) FROM DIM_COURT
UNION ALL
SELECT 'DIM_BEST_OF', COUNT(*) FROM DIM_BEST_OF
UNION ALL
SELECT 'DIM_PLAYER', COUNT(*) FROM DIM_PLAYER
UNION ALL
SELECT 'FACT_MATCH', COUNT(*) FROM FACT_MATCH;
""")


,table_name,row_count
0,DIM_DATE,6609
1,DIM_TOURNAMENT,331
2,DIM_ROUND,8
3,DIM_SURFACE,4
4,DIM_COURT,2
5,DIM_BEST_OF,2
6,DIM_PLAYER,1798
7,FACT_MATCH,67512


### 5.3 Final measure checks

This check validates the analytical measures used for upset analysis.


In [52]:
q("""
SELECT
    SUM(match_count) AS total_matches,
    SUM(upset_count) AS upset_matches,
    SUM(non_upset_count) AS non_upset_matches,
    SUM(observable_upset_count) AS observable_upset_matches,
    SUM(missing_upset_flag_count) AS non_observable_upset_matches,
    ROUND(
        1.0 * SUM(upset_count) / NULLIF(SUM(observable_upset_count), 0),
        4
    ) AS upset_rate
FROM FACT_MATCH;
""")

,total_matches,upset_matches,non_upset_matches,observable_upset_matches,non_observable_upset_matches,upset_rate
0,67512,18607,43694,62301,5211,0.2987


In [53]:
q("""
SELECT
    SUM(observable_upset_count) AS observable,
    SUM(upset_count) + SUM(non_upset_count) AS upset_plus_non_upset,
    SUM(observable_upset_count) - (SUM(upset_count) + SUM(non_upset_count)) AS difference
FROM FACT_MATCH;
""")

,observable,upset_plus_non_upset,difference
0,62301,62301,0


### 5.4 Foreign-key validation

This validation checks whether the surrogate keys stored in `FACT_MATCH` correctly join to the related dimension tables. This is different from checking row counts: the fact table may contain the correct number of rows while still having missing or incorrect dimension links.

Favorite and underdog keys are expected to be missing only for non-observable odds-based cases.


In [54]:
q("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN date_key IS NULL THEN 1 ELSE 0 END) AS missing_date_key,
    SUM(CASE WHEN tournament_key IS NULL THEN 1 ELSE 0 END) AS missing_tournament_key,
    SUM(CASE WHEN round_key IS NULL THEN 1 ELSE 0 END) AS missing_round_key,
    SUM(CASE WHEN surface_key IS NULL THEN 1 ELSE 0 END) AS missing_surface_key,
    SUM(CASE WHEN court_key IS NULL THEN 1 ELSE 0 END) AS missing_court_key,
    SUM(CASE WHEN best_of_key IS NULL THEN 1 ELSE 0 END) AS missing_best_of_key,
    SUM(CASE WHEN winner_player_key IS NULL THEN 1 ELSE 0 END) AS missing_winner_key,
    SUM(CASE WHEN loser_player_key IS NULL THEN 1 ELSE 0 END) AS missing_loser_key,
    SUM(CASE WHEN favorite_player_key IS NULL THEN 1 ELSE 0 END) AS missing_favorite_key,
    SUM(CASE WHEN underdog_player_key IS NULL THEN 1 ELSE 0 END) AS missing_underdog_key
FROM FACT_MATCH;
""")

,total_rows,missing_date_key,missing_tournament_key,missing_round_key,missing_surface_key,missing_court_key,missing_best_of_key,missing_winner_key,missing_loser_key,missing_favorite_key,missing_underdog_key
0,67512,0,0,0,0,0,0,0,0,5211,5211


## 6. Export for Tableau

The data warehouse is exported to CSV files. A flat CSV file is also created for Tableau Public, which does not support direct SQLite database connections.


In [55]:
import os

export_dir = "exports"
os.makedirs(export_dir, exist_ok=True)

In [56]:
for table in [
    "DIM_DATE", "DIM_TOURNAMENT", "DIM_ROUND", "DIM_SURFACE",
    "DIM_COURT", "DIM_BEST_OF", "DIM_PLAYER", "FACT_MATCH"
]:
    df = q(f"SELECT * FROM {table};")
    df.to_csv(f"{export_dir}/{table}.csv", index=False)
    print(f"Exported {table}.csv with {len(df)} rows")


Exported DIM_DATE.csv with 6609 rows
Exported DIM_TOURNAMENT.csv with 331 rows
Exported DIM_ROUND.csv with 8 rows
Exported DIM_SURFACE.csv with 4 rows
Exported DIM_COURT.csv with 2 rows
Exported DIM_BEST_OF.csv with 2 rows
Exported DIM_PLAYER.csv with 1798 rows
Exported FACT_MATCH.csv with 67512 rows


In [57]:
tableau_flat = q("""
SELECT
    f.match_key,

    d.match_date,
    CAST(strftime('%d', d.match_date) AS INTEGER) AS day,
    CAST(strftime('%m', d.match_date) AS INTEGER) AS month,
    CASE
        WHEN CAST(strftime('%m', d.match_date) AS INTEGER) BETWEEN 1 AND 3 THEN 1
        WHEN CAST(strftime('%m', d.match_date) AS INTEGER) BETWEEN 4 AND 6 THEN 2
        WHEN CAST(strftime('%m', d.match_date) AS INTEGER) BETWEEN 7 AND 9 THEN 3
        ELSE 4
    END AS quarter,
    CAST(strftime('%Y', d.match_date) AS INTEGER) AS year,

    t.tournament_name,
    t.series,

    r.round_name,
    r.round_order,

    s.surface,
    c.court,
    b.best_of,

    wp.player_name AS winner_name,
    lp.player_name AS loser_name,
    fp.player_name AS favorite_name,
    up.player_name AS underdog_name,

    f.match_count,
    f.upset_count,
    f.non_upset_count,
    f.observable_upset_count,
    f.missing_upset_flag_count,
    f.equal_odds_count,

    f.winner_rank,
    f.loser_rank,
    f.rank_gap,

    f.winner_points,
    f.loser_points,
    f.points_gap,

    f.winner_odds,
    f.loser_odds,
    f.favorite_odds,
    f.underdog_odds,
    f.odd_gap

FROM FACT_MATCH f
LEFT JOIN DIM_DATE d
    ON f.date_key = d.date_key
LEFT JOIN DIM_TOURNAMENT t
    ON f.tournament_key = t.tournament_key
LEFT JOIN DIM_ROUND r
    ON f.round_key = r.round_key
LEFT JOIN DIM_SURFACE s
    ON f.surface_key = s.surface_key
LEFT JOIN DIM_COURT c
    ON f.court_key = c.court_key
LEFT JOIN DIM_BEST_OF b
    ON f.best_of_key = b.best_of_key
LEFT JOIN DIM_PLAYER wp
    ON f.winner_player_key = wp.player_key
LEFT JOIN DIM_PLAYER lp
    ON f.loser_player_key = lp.player_key
LEFT JOIN DIM_PLAYER fp
    ON f.favorite_player_key = fp.player_key
LEFT JOIN DIM_PLAYER up
    ON f.underdog_player_key = up.player_key;
""")

tableau_flat.to_csv("exports/tableau_match_flat.csv", index=False)

tableau_flat.head()

,match_key,match_date,day,month,quarter,year,tournament_name,series,round_name,round_order,...,loser_rank,rank_gap,winner_points,loser_points,points_gap,winner_odds,loser_odds,favorite_odds,underdog_odds,odd_gap
0,1,2000-01-03,3,1,1,2000,Australian Hardcourt Championships,International,1st Round,1,...,77.0,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,2000-01-03,3,1,1,2000,Australian Hardcourt Championships,International,1st Round,1,...,56.0,51.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,2000-01-03,3,1,1,2000,Australian Hardcourt Championships,International,1st Round,1,...,655.0,615.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,2000-01-03,3,1,1,2000,Australian Hardcourt Championships,International,1st Round,1,...,87.0,22.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,2000-01-03,3,1,1,2000,Australian Hardcourt Championships,International,1st Round,1,...,198.0,117.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [58]:
tableau_flat.shape

(67512, 34)

### 6.1 Tableau export validation

The final flat export must preserve the transactional granularity of one row per match. Mandatory descriptive fields should be populated. `favorite_name` and `underdog_name` may be NULL for the 5,211 non-observable odds-based cases.


In [59]:
tableau_flat.isna().sum()

,0
match_key,0
match_date,0
day,0
month,0
quarter,0
year,0
tournament_name,0
series,0
round_name,0
round_order,0


In [60]:
import os

os.makedirs("exports", exist_ok=True)

tableau_flat.to_csv("exports/tableau_match_flat.csv", index=False)

## 7. Conclusion

Phase 2 is completed.

The reconciled database was assessed through checks on population, completeness, uniqueness, consistency and numerical validity. The main data quality issues concern missing or unreliable analytical attributes, especially betting odds and ranking points.

The adopted cleaning strategy preserves valid match records, avoids artificial imputations and introduces analytical flags to distinguish observable and non-observable cases.

The ETL process populates the star-schema data warehouse and validates row counts, measure consistency and mandatory foreign-key completeness.

Finally, the warehouse is exported into a Tableau-ready flat CSV file containing 67,512 rows and 34 columns.


In [61]:
!zip -r tennis_dw_tableau_exports.zip exports

  adding: exports/ (stored 0%)
  adding: exports/DIM_BEST_OF.csv (deflated 11%)
  adding: exports/tableau_match_flat.csv (deflated 84%)
  adding: exports/DIM_DATE.csv (deflated 86%)
  adding: exports/DIM_PLAYER.csv (deflated 54%)
  adding: exports/FACT_MATCH.csv (deflated 77%)
  adding: exports/DIM_SURFACE.csv (deflated 6%)
  adding: exports/DIM_ROUND.csv (deflated 33%)
  adding: exports/DIM_COURT.csv (deflated 11%)
  adding: exports/DIM_TOURNAMENT.csv (deflated 67%)


In [62]:
from google.colab import files
files.download("tennis_dw_tableau_exports.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [63]:
final_health_check = q("""
SELECT
    'FACT row count' AS check_name,
    CASE WHEN COUNT(*) = 67512 THEN 'OK' ELSE 'ERROR' END AS status,
    COUNT(*) AS value
FROM FACT_MATCH

UNION ALL

SELECT
    'Observable consistency',
    CASE
        WHEN SUM(observable_upset_count) = SUM(upset_count) + SUM(non_upset_count)
        THEN 'OK' ELSE 'ERROR'
    END,
    SUM(observable_upset_count) - (SUM(upset_count) + SUM(non_upset_count))
FROM FACT_MATCH

UNION ALL

SELECT
    'Non observable consistency',
    CASE
        WHEN SUM(observable_upset_count) + SUM(missing_upset_flag_count) = SUM(match_count)
        THEN 'OK' ELSE 'ERROR'
    END,
    SUM(match_count) - (SUM(observable_upset_count) + SUM(missing_upset_flag_count))
FROM FACT_MATCH

UNION ALL

SELECT
    'Mandatory FK missing',
    CASE
        WHEN
            SUM(CASE WHEN date_key IS NULL THEN 1 ELSE 0 END) +
            SUM(CASE WHEN tournament_key IS NULL THEN 1 ELSE 0 END) +
            SUM(CASE WHEN round_key IS NULL THEN 1 ELSE 0 END) +
            SUM(CASE WHEN surface_key IS NULL THEN 1 ELSE 0 END) +
            SUM(CASE WHEN court_key IS NULL THEN 1 ELSE 0 END) +
            SUM(CASE WHEN best_of_key IS NULL THEN 1 ELSE 0 END) +
            SUM(CASE WHEN winner_player_key IS NULL THEN 1 ELSE 0 END) +
            SUM(CASE WHEN loser_player_key IS NULL THEN 1 ELSE 0 END)
        = 0
        THEN 'OK' ELSE 'ERROR'
    END,
    SUM(CASE WHEN date_key IS NULL THEN 1 ELSE 0 END) +
    SUM(CASE WHEN tournament_key IS NULL THEN 1 ELSE 0 END) +
    SUM(CASE WHEN round_key IS NULL THEN 1 ELSE 0 END) +
    SUM(CASE WHEN surface_key IS NULL THEN 1 ELSE 0 END) +
    SUM(CASE WHEN court_key IS NULL THEN 1 ELSE 0 END) +
    SUM(CASE WHEN best_of_key IS NULL THEN 1 ELSE 0 END) +
    SUM(CASE WHEN winner_player_key IS NULL THEN 1 ELSE 0 END) +
    SUM(CASE WHEN loser_player_key IS NULL THEN 1 ELSE 0 END)
FROM FACT_MATCH
""")

final_health_check

,check_name,status,value
0,FACT row count,OK,67512
1,Observable consistency,OK,0
2,Non observable consistency,OK,0
3,Mandatory FK missing,OK,0
